# **PyTesseract ve EASY OCR ile Karakter Tanıma**

- Bu derste PyTesseract kullanarak birkaç görüntü üzerinde OCR uygulayacağız
- Ayrıca Easy OCR kullanacağız

<img src="OCR.png" width="600">

OCR, Optical Character Recognition (Optik Karakter Tanıma) \
📌 Tanımı:\
Bir görüntüdeki (resim, taranmış belge, PDF vb.) yazıların bilgisayar tarafından okunup dijital, düzenlenebilir metne dönüştürülmesini sağlayan teknolojidir.\
Kullanım Alanları\
📑 Taranmış belgeleri Word veya PDF’e dönüştürmek\
📷 Fotoğraftaki yazıları almak (ör. ekran görüntüsü, fatura, tabela)\
🏦 Bankacılıkta çek/IBAN tanıma\
🚗 Plaka tanıma sistemleri\
📚 Eski kitapları dijitalleştirme

**“Tesseract”, İngilizcede dördüncü boyuttaki küpün (4D hypercube) adıdır.**\
Google’ın geliştirdiği Tesseract OCR yazılımına bu ismin verilmesinin sebebi:\
OCR probleminin karmaşık ve çok boyutlu bir problem olması,\
Yazının farklı font, boyut, eğim ve gürültü gibi birçok değişkeninin (boyutunun) olmasıdır.

# PyTesseract yükleyelim

In [ ]:
# PyTesseract'ı yükleyelim
!pip install pytesseract

# Tesseract yükleyelim
https://github.com/UB-Mannheim/tesseract/wiki


In [ ]:
import cv2
import pytesseract
import numpy as np
from matplotlib import pyplot as plt

pytesseract.pytesseract.tesseract_cmd = (
    r'/usr/bin/tesseract'
)

def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()


## **İlk OCR Denemesi**

In [ ]:
# Öncelikle hata vermemesi için pytesseract yüklenen yolu bilgisayarımıza gösterelim
import pytesseract
# Başka bir yere kurulduysa aşağıdaki yolu güncelleyin
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"


In [ ]:
img = cv2.imread('../files/images/OCR1.png')
imshow("Input Image", img)

# PyTesseract aracılığıyla görüntümüzü analiz edelim
output_txt = pytesseract.image_to_string(img)

print("PyTesseract Extracted: {}".format(output_txt))

## **Siyah zeminler üzerinde beyaz metin okunabilir mi?**

In [ ]:
img = cv2.imread('../files/images/OCR2.png')
imshow("Input Image", img)

# PyTesseract aracılığıyla görüntümüzü analiz edelim
output_txt = pytesseract.image_to_string(img)

print("PyTesseract Extracted: {}".format(output_txt))

## **Daha karmaşık arka planlarda okuma**

In [ ]:
img = cv2.imread('../files/images/OCR3.png')
imshow("Input Image", img)

# PyTesseract aracılığıyla görüntümüzü analiz edelim
output_txt = pytesseract.image_to_string(img)

print("PyTesseract Extracted: {}".format(output_txt))

## **What about a real life scan?**

In [ ]:
img = cv2.imread('../files/images/scan2.jpeg')
imshow("Input Image", img, size = 48)

# PyTesseract aracılığıyla görüntümüzü analiz edelim
output_txt = pytesseract.image_to_string(img)

print("PyTesseract Extracted: {}".format(output_txt))

# **Resimde Düzeltme Yaparak**

In [ ]:
from skimage.filters import threshold_local

image = cv2.imread('../files/images/scan2.jpeg')
imshow("Input Image", image, size = 48)

# Değer bileşenini HSV renk uzayından alıyoruz 
# sonra uyarlamalı eşikleme uyguluyoruz
V = cv2.split(cv2.cvtColor(image, cv2.COLOR_BGR2HSV))[2]
T = threshold_local(V, 25, offset=15, method="gaussian")

# Eşik işlemini uygulayalım
thresh = (V > T).astype("uint8") * 255
imshow("threshold_local", thresh, size = 48)

output_txt = pytesseract.image_to_string(thresh)
print("PyTesseract Extracted: {}".format(output_txt))

### **Eşikleme Daha iyi okuma sağlar**

Tipik olarak OCR tanıma için iyi bir ön işleme hattı aşağıdaki işlemlerden bazılarını veya daha fazlasını içerecektir:
1. Bluring (Bulanıklaştırma)
2. Thresholding (Eşkileme)
3. Deskewing (Eğim Düzeltme)
4. Dilation/Erosion/Opening/Closing (Dilatasyon/Erozyon/Açılma/Kapanma)
5. Noise Removal (Gürültü Giderme)

### **PyTesseract Tarafından Tanınan Bölgeler Üzerinde Çizim Yapalım**

In [ ]:
from skimage.filters import threshold_local

image = cv2.imread('../files/images/Receipt-woolworth.jpg')

# We get the Value component from the HSV color space 
# then we apply adaptive thresholdingto 
V = cv2.split(cv2.cvtColor(image, cv2.COLOR_BGR2HSV))[2]
T = threshold_local(V, 25, offset=15, method="gaussian")

# Apply the threshold operation 
thresh = (V > T).astype("uint8") * 255
imshow("threshold_local", thresh,6)

output_txt = pytesseract.image_to_string(thresh)
print("PyTesseract Extracted: {}".format(output_txt))

In [ ]:
from pytesseract import Output

d = pytesseract.image_to_data(thresh, output_type = Output.DICT)
print(d.keys())

### Bu sözlüğü kullanarak, tespit edilen her kelimeyi, sınırlayıcı kutu bilgilerini, içlerindeki metni ve her biri için güven puanlarını alabiliriz.


In [ ]:
n_boxes = len(d['text'])

for i in range(n_boxes):
    if int(d['conf'][i]) > 60:
        (x, y, w, h) = (d['left'][i], d['top'][i], d['width'][i], d['height'][i])
        image = cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)

imshow('Output', image, size = 6)


## **EASY OCR**

In [ ]:
!pip install easyocr

**Resimdeki Metni Algıla ve Girdi Resmini Göster**

In [ ]:
# gerekli paketleri içe aktaralım
from matplotlib import pyplot as plt
from easyocr import Reader
import pandas as pd
import cv2
import time

# imshow fonksiyonumuzu tanımlayalım
def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

# girdi görüntüsünü yükleyelim
image = cv2.imread("../files/images/whatsapp_conv.jpeg")
imshow("Original Image", image, size = 12)

# EasyOCR kullanarak giriş görüntüsünü OCR
print("Detecting and OCR'ing text from input image...")
reader = Reader(['en'], gpu = False)

ts = time.time()
results = reader.readtext(image)
te = time.time()
td = te - ts
print(f'Completed in {td} seconds')
print("Tamamlandı")

In [ ]:
results

## **Resmimizin Üzerine Yerleştirilmiş Metni Göster**

In [ ]:
all_text = []

# iterate over our extracted text 
for (bbox, text, prob) in results:
    # display the OCR'd text and the associated probability of it being text
    print(f" Probability of Text: {prob*100:.3f}% OCR'd Text: {text}")

    # get the bounding box coordinates
    (tl, tr, br, bl) = bbox
    tl = (int(tl[0]), int(tl[1]))
    tr = (int(tr[0]), int(tr[1]))
    br = (int(br[0]), int(br[1]))
    bl = (int(bl[0]), int(bl[1]))

    # Remove non-ASCII characters from the text so that
    # we can draw the box surrounding the text overlaid onto the original image
    text = "".join([c if ord(c) < 128 else "" for c in text]).strip()
    all_text.append(text)
    cv2.rectangle(image, tl, br, (255, 0, 0), 2)
    cv2.putText(image, text, (tl[0], tl[1] - 10),
      cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

# show the output image
imshow("OCR'd Image", image, size = 12)

## **WoolWorth Reciept'i üzerinde çalıştıralım**

In [ ]:
import cv2
from easyocr import Reader
import numpy as np
from matplotlib import pyplot as plt

# imshow fonksiyonumuzu tanımlayın 
def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

def clean_text(text):
	# ASCII olmayan metni kaldıalım, böylece metni görüntü üzerine yazabiliriz
	return "".join([c if ord(c) < 128 else "" for c in text]).strip()

image = cv2.imread('../files/images/Receipt-woolworth.jpg')

reader = Reader(["en","ar"], gpu=False)
results = reader.readtext(image)

# sonuçlar üzerinde döngü
for (bbox, text, prob) in results:
	# OCR'lanan metni ve olasılığı görüntüleyelim
	print("[INFO] {:.4f}: {}".format(prob, text))

	# sınırlayıcı kutuyu işleyelim
	(tl, tr, br, bl) = bbox
	tl = (int(tl[0]), int(tl[1]))
	tr = (int(tr[0]), int(tr[1]))
	br = (int(br[0]), int(br[1]))
	bl = (int(bl[0]), int(bl[1]))

	# metni temizleyin ve metni çevreleyen kutuyu birlikte çizin
	text = clean_text(text)
	cv2.rectangle(image, tl, br, (0, 255, 0), 2)
	cv2.putText(image, text, (tl[0], tl[1] - 10),
		cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

# Eşik işlemini uygulayalım
#thresh = (V > T).astype("uint8") * 255
imshow("EASY OCR", image)
print("EASY OCR Extracted: {}".format(text))